# Cleaning & merging the Bitcoin price data

**Project:** BITCOIN-TREND-PROJECT
**Notebook goal:** take the two messy raw CSVs I downloaded and turn them into one clean, continuous daily dataset I can actually train a model on.

I'm writing this notebook as my "lab notebook" so future-me (and whoever grades this) can see *why* I did each step, not just *what* I did. The actual cleaning logic also lives in `clean_btc_data.py` in the project root — this notebook is the explained version.

## The short story
I have **two** sources of Bitcoin daily prices:

1. **CryptoDataDownload** file — covers **2014 → 2022**. It's the only place I have the *early* history (2014–2017).
2. **Binance** file — covers **2018 → 2026**. Higher quality, more recent, and goes almost up to today.

They overlap (2018–2022), they have **different column names and even different numbers of columns**, and the old file has a sneaky **bug in its volume columns**. So the job is: line them up, fix the bug, stitch them together without double-counting the overlap, and end up with one tidy table.

The final table has only **8 columns**, even though my raw files had 9 and 12. A big part of this notebook explains *where all those columns went and why I dropped them* — that part confused me at first, so I wrote it out in detail.

In [1]:
import os
from pathlib import Path

import pandas as pd

# Find the project root by climbing up until we see the `data/` folder.
# This way the notebook runs whether it's in the project root OR in notebooks/.
def find_project_root(marker="data"):
    here = Path.cwd()
    for parent in [here, *here.parents]:
        if (parent / marker).is_dir():
            return parent
    raise FileNotFoundError("Could not find the 'data' folder above the current directory.")

ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CRYPTODATADOWNLOAD = RAW_DIR / "btc_usd_daily_2014_2022_cryptodatadownload.csv"
BINANCE = RAW_DIR / "btc_usd_daily_2018_2026_binance.csv"
OUT = PROCESSED_DIR / "btc_usd_daily_2014_2026.csv"

print("Project root:", ROOT)
print("Reading raw from:", RAW_DIR)

Project root: D:\Bitcoin-Trend-Prediction
Reading raw from: D:\Bitcoin-Trend-Prediction\data\raw


## 1. Look at the raw Binance file first

Rule I'm trying to follow: **never clean data I haven't actually looked at.** So before touching anything, I just load each file and stare at it — shape, column names, first few rows, data types.

I'm starting with Binance because it's the bigger, more modern, more trustworthy file, and it'll be the "authoritative" source for everything from 2018 onward.

In [2]:
raw_binance = pd.read_csv(BINANCE)
print("Shape (rows, cols):", raw_binance.shape)
print("\nColumns:")
for c in raw_binance.columns:
    print("  -", c)
raw_binance.head()

Shape (rows, cols): (3089, 12)

Columns:
  - Open time
  - Open
  - High
  - Low
  - Close
  - Volume
  - Close time
  - Quote asset volume
  - Number of trades
  - Taker buy base asset volume
  - Taker buy quote asset volume
  - Ignore


,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore
0,2018-01-01 00:00:00.000000 UTC,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000 UTC,1.147997e+08,105595,3961.938946,5.280975e+07,0
1,2018-01-02 00:00:00.000000 UTC,13382.16,15473.49,12890.02,14675.11,20078.092111,2018-01-02 23:59:59.999000 UTC,2.797171e+08,177728,11346.326739,1.580801e+08,0
2,2018-01-03 00:00:00.000000 UTC,14690.00,15307.56,14150.00,14919.51,15905.667639,2018-01-03 23:59:59.999000 UTC,2.361169e+08,162787,8994.953566,1.335873e+08,0
3,2018-01-04 00:00:00.000000 UTC,14919.51,15280.00,13918.04,15059.54,21329.649574,2018-01-04 23:59:59.999000 UTC,3.127816e+08,170310,12680.812951,1.861168e+08,0
4,2018-01-05 00:00:00.000000 UTC,15059.56,17176.24,14600.00,16960.39,23251.491125,2018-01-05 23:59:59.999000 UTC,3.693220e+08,192969,13346.622293,2.118299e+08,0


In [3]:
# What types did pandas guess, and is anything missing?
raw_binance.info()

<class 'pandas.DataFrame'>
RangeIndex: 3089 entries, 0 to 3088
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Open time                     3089 non-null   str    
 1   Open                          3089 non-null   float64
 2   High                          3089 non-null   float64
 3   Low                           3089 non-null   float64
 4   Close                         3089 non-null   float64
 5   Volume                        3089 non-null   float64
 6   Close time                    3089 non-null   str    
 7   Quote asset volume            3089 non-null   float64
 8   Number of trades              3089 non-null   int64  
 9   Taker buy base asset volume   3089 non-null   float64
 10  Taker buy quote asset volume  3089 non-null   float64
 11  Ignore                        3089 non-null   int64  
dtypes: float64(8), int64(2), str(2)
memory usage: 289.7 KB


### What are all these Binance columns, and which do I keep?

Binance gives **12 columns**. That's a lot more than I need. This is "exchange kline (candlestick) format" — it's built for traders, not for a simple price model. Here's my reasoning column by column:

| Raw Binance column | Keep? | Renamed to | Why |
|---|---|---|---|
| `Open time` | ✅ | `date` | The day this candle covers. I'll turn it into a clean date. |
| `Open` | ✅ | `open` | Price at the start of the day. |
| `High` | ✅ | `high` | Highest price that day. |
| `Low` | ✅ | `low` | Lowest price that day. |
| `Close` | ✅ | `close` | Price at the end of the day — the most important one. |
| `Volume` | ✅ | `volume_btc` | How much **BTC** was traded (base-asset volume). |
| `Quote asset volume` | ✅ | `volume_usd` | How much **USD** changed hands (quote-asset volume). |
| `Close time` | ❌ | — | Redundant. For *daily* candles it's always `Open time` + ~24h. No new info. |
| `Number of trades` | ❌ | — | Useful in theory, but the **old file doesn't have it**, so I can't have it for 2014–2017 (more on this below). |
| `Taker buy base asset volume` | ❌ | — | Order-flow detail. Binance-only → can't span the full history. |
| `Taker buy quote asset volume` | ❌ | — | Same problem. |
| `Ignore` | ❌ | — | This is a literal placeholder column Binance ships that's always `0`. It's even *named* "Ignore." Easy drop. |

So I'm going from **12 → 7** columns from this file (plus I'll add a `source` tag later).

**Quick sanity check on the two volume columns** (this matters for the bug later): USD volume should roughly equal BTC volume × price. Let me verify on the first row so I'm sure I've got `volume_btc` vs `volume_usd` the right way round.

In [4]:
r = raw_binance.iloc[0]
print("Volume (BTC) x Close = %.0f" % (r["Volume"] * r["Close"]))
print("Quote asset volume   = %.0f" % r["Quote asset volume"])
print("-> these match, so Volume = BTC quantity and Quote asset volume = USD value. Good.")

Volume (BTC) x Close = 115200674
Quote asset volume   = 114799747
-> these match, so Volume = BTC quantity and Quote asset volume = USD value. Good.


## 2. Now look at the raw CryptoDataDownload file

This is the older file. It's the **only** place I have 2014–2017 data, so even though it's lower quality, I need it for the early history.

In [5]:
raw_cdd = pd.read_csv(CRYPTODATADOWNLOAD)
print("Shape (rows, cols):", raw_cdd.shape)
print("\nColumns:")
for c in raw_cdd.columns:
    print("  -", c)
raw_cdd.head()

Shape (rows, cols): (2651, 9)

Columns:
  - unix
  - date
  - symbol
  - open
  - high
  - low
  - close
  - Volume BTC
  - Volume USD


,unix,date,symbol,open,high,low,close,Volume BTC,Volume USD
0,1646092800,2022-03-01 00:00:00,BTC/USD,43221.71,43626.49,43185.48,43185.48,49.006289,2.116360e+06
1,1646006400,2022-02-28 00:00:00,BTC/USD,37717.10,44256.08,37468.99,43178.98,3160.618070,1.364723e+08
2,1645920000,2022-02-27 00:00:00,BTC/USD,39146.66,39886.92,37015.74,37712.68,1701.817043,6.418008e+07
3,1645833600,2022-02-26 00:00:00,BTC/USD,39242.64,40330.99,38600.00,39146.66,912.724087,3.573010e+07
4,1645747200,2022-02-25 00:00:00,BTC/USD,38360.93,39727.97,38027.61,39231.64,2202.851827,8.642149e+07


### What are the CryptoDataDownload columns, and which do I keep?

This file has **9 columns**. Different names from Binance (lowercase, different volume labels), and a couple of columns that carry no real information:

| Raw column | Keep? | Renamed to | Why |
|---|---|---|---|
| `unix` | ❌ | — | This is just the date as a Unix timestamp (seconds since 1970). It's the **same information** as `date`, only less readable. Keeping both is redundant. |
| `date` | ✅ | `date` | Human-readable timestamp. I'll keep this one. |
| `symbol` | ❌ | — | It says `BTC/USD` on **every single row**. A column that never changes tells the model nothing — zero information, so drop it. |
| `open` | ✅ | `open` | OHLC. |
| `high` | ✅ | `high` | OHLC. |
| `low` | ✅ | `low` | OHLC. |
| `close` | ✅ | `close` | OHLC. |
| `Volume BTC` | ✅* | `volume_btc` | BTC traded. **\\*but the values are buggy — fixed in step 4.** |
| `Volume USD` | ✅* | `volume_usd` | USD traded. **\\*same bug.** |

So this file goes **9 → 7** columns.

## 3. So… where did all the columns go? (the big question)

My raw files had **9** and **12** columns. My final dataset has **8**. Here's the honest accounting of every dropped column:

**Dropped because they're redundant / empty:**
- `unix` → same as `date`, just uglier.
- `symbol` → constant `BTC/USD`, no information.
- `Close time` → for daily data it's just end-of-day, derivable from the date.
- `Ignore` → Binance's always-zero placeholder.

**Dropped because of the merge — this is the important lesson:**
- `Number of trades`, `Taker buy base asset volume`, `Taker buy quote asset volume`

These three are genuinely useful trading features… but **only Binance has them**. The CryptoDataDownload file (my 2014–2017 history) doesn't. 

Here's the rule I learned: **when you merge two datasets into one continuous table, you can only keep columns that BOTH sources can fill.** If I kept `Number of trades`, then every row from 2014–2017 would be blank (`NaN`) in that column. A model would either crash on those gaps or I'd be forced to throw away the early years — which defeats the whole point of merging the old file in. 

This is sometimes called keeping the **"lowest common denominator"** of columns: the *intersection* of what both files provide. I'm trading away some nice Binance-only features in exchange for a clean, gap-free dataset that spans 12 years. For a first model, that trade is worth it. (If I later decide those order-flow features matter, I'd build a *separate* Binance-only dataset for the 2018+ period — but that's a different experiment.)

**Added (didn't exist in either file):**
- `source` → I tag each row `binance` or `early-history` so I can always see where it came from. Cheap insurance and good for debugging.

**Final 8 columns:** `date, open, high, low, close, volume_btc, volume_usd, source`.

Now I'll actually do the cleaning.

## 4. Clean the Binance file (rename, select, fix the date)

Three small operations:
1. Convert `Open time` into a real datetime, strip the timezone, and "normalize" it (chop off the time-of-day so it's a clean date at midnight).
2. Rename the columns to my lowercase canonical names.
3. Keep only the 7 columns I decided on, and tag the source.

In [6]:
new = pd.read_csv(BINANCE)

# 1. Open time -> clean date (drop timezone, keep just the day)
new["date"] = pd.to_datetime(new["Open time"]).dt.tz_localize(None).dt.normalize()

# 2. Rename to canonical names
new = new.rename(columns={
    "Open": "open", "High": "high", "Low": "low", "Close": "close",
    "Volume": "volume_btc", "Quote asset volume": "volume_usd",
})

# 3. Keep only what I need + tag the source
new = new[["date", "open", "high", "low", "close", "volume_btc", "volume_usd"]].copy()
new["source"] = "binance"

print("Binance after cleaning:", new.shape)
new.head()

Binance after cleaning: (3089, 8)


,date,open,high,low,close,volume_btc,volume_usd,source
0,2018-01-01,13715.65,13818.55,12750.00,13380.00,8609.915844,1.147997e+08,binance
1,2018-01-02,13382.16,15473.49,12890.02,14675.11,20078.092111,2.797171e+08,binance
2,2018-01-03,14690.00,15307.56,14150.00,14919.51,15905.667639,2.361169e+08,binance
3,2018-01-04,14919.51,15280.00,13918.04,15059.54,21329.649574,3.127816e+08,binance
4,2018-01-05,15059.56,17176.24,14600.00,16960.39,23251.491125,3.693220e+08,binance


## 5. The CryptoDataDownload volume bug

This one took me a while to spot. Look at an early row from the old file (2014) versus a late row (2022):

In [7]:
old_raw = pd.read_csv(CRYPTODATADOWNLOAD)
old_raw["date"] = pd.to_datetime(old_raw["date"])
print("A 2014 row:")
print(old_raw.sort_values("date").iloc[0][["date", "close", "Volume BTC", "Volume USD"]])
print("\nA 2022 row:")
print(old_raw.sort_values("date").iloc[-1][["date", "close", "Volume BTC", "Volume USD"]])

A 2014 row:


date          2014-11-28 00:00:00
close                      376.28
Volume BTC             3220878.18
Volume USD                8617.15
Name: 2650, dtype: object

A 2022 row:
date          2022-03-01 00:00:00
close                    43185.48
Volume BTC              49.006289
Volume USD         2116360.100528
Name: 0, dtype: object


### Why this is obviously wrong

In the **2022** row the numbers make sense: a small `Volume BTC` and a large `Volume USD`, and `Volume BTC × close ≈ Volume USD`. That's the relationship that *must* hold — the USD value traded equals the amount of BTC traded times the price.

In the **2014** row it's backwards. `Volume BTC` is in the millions and `Volume USD` is tiny. If I took that literally, it'd mean millions of BTC (worth over a billion dollars) traded in a single day in 2014, while only a few thousand dollars changed hands. That's impossible — the two columns are **swapped** in the early rows.

### How I fix it (without hardcoding a cutoff date)
Instead of guessing where the swap starts, I check **every row** against the rule "USD ≈ BTC × price" and decide per-row which labeling is correct:

- If `Volume BTC × close ≈ Volume USD` → labels are **correct**, leave it.
- If `Volume USD × close ≈ Volume BTC` → labels are **swapped**, flip them.

I compare the relative error of each interpretation and pick whichever fits better. This is more robust than assuming a clean boundary, because it self-corrects on a row-by-row basis.

In [8]:
a = old_raw["Volume BTC"].astype(float)   # column AS LABELED (often wrong early on)
b = old_raw["Volume USD"].astype(float)
p = old_raw["close"].astype(float)

# Relative error of each interpretation
err_correct = (a * p - b).abs() / b.replace(0, 1).abs()   # assumes a=BTC, b=USD
err_swapped = (b * p - a).abs() / a.replace(0, 1).abs()   # assumes b=BTC, a=USD
swapped = err_swapped < err_correct                        # True where labels are flipped

old_raw["volume_btc"] = a.where(~swapped, b)   # use b when swapped
old_raw["volume_usd"] = b.where(~swapped, a)   # use a when swapped

print(f"Fixed {int(swapped.sum())} swapped-volume rows out of {len(old_raw)}.")
print("\nThe same 2014 row, now corrected:")
print(old_raw.sort_values('date').iloc[0][['date', 'close', 'volume_btc', 'volume_usd']])

Fixed 1180 swapped-volume rows out of 2651.

The same 2014 row, now corrected:
date          2014-11-28 00:00:00
close                      376.28
volume_btc                8617.15
volume_usd             3220878.18
Name: 2650, dtype: object


## 6. Keep only the early history from the old file

The old file runs 2014–2022, but Binance already covers 2018 onward **and** does it better (no volume bug, cleaner data). So I only want the old file for the part Binance *doesn't* have: everything **before 2018-01-01**.

This also neatly avoids the overlap problem — I won't have two versions of, say, 2019 fighting each other.

In [9]:
CUTOFF = pd.Timestamp("2018-01-01")  # Binance is authoritative from here on

old = old_raw[["date", "open", "high", "low", "close", "volume_btc", "volume_usd"]].copy()
old["date"] = old["date"].dt.normalize()
old["source"] = "early-history"

old_early = old[old["date"] < CUTOFF]
print("Keeping early-history rows before 2018:", old_early.shape)
print("Early range:", old_early["date"].min().date(), "->", old_early["date"].max().date())

Keeping early-history rows before 2018: (1130, 8)
Early range: 2014-11-28 -> 2017-12-31


## 7. Merge, remove duplicates, sort

Now I stack the two pieces together:
- `old_early` (2014 → 2017, from the old file)
- `new` (2018 → 2026, from Binance)

I `drop_duplicates` on the date just as a safety net (`keep="last"` means if a date somehow appears in both, the Binance version wins, because I concatenated it second). Then I sort by date so the timeline runs oldest → newest, and reset the index so it's clean `0, 1, 2, …`.

In [10]:
merged = pd.concat([old_early, new], ignore_index=True)
merged = merged.drop_duplicates(subset="date", keep="last")  # prefer Binance on any overlap
merged = merged.sort_values("date").reset_index(drop=True)

print("Merged shape:", merged.shape)
merged.head()

Merged shape: (4219, 8)


,date,open,high,low,close,volume_btc,volume_usd,source
0,2014-11-28,363.59,381.34,360.57,376.28,8617.15,3220878.18,early-history
1,2014-11-29,376.42,386.60,372.25,376.72,7245.19,2746157.05,early-history
2,2014-11-30,376.57,381.99,373.32,373.34,3046.33,1145566.61,early-history
3,2014-12-01,376.40,382.31,373.03,378.39,6660.56,2520662.37,early-history
4,2014-12-02,378.39,382.86,375.23,379.25,6832.53,2593576.46,early-history


## 8. Validation — don't trust it until I've checked it

Cleaning code can run perfectly and still produce garbage, so I run a checklist before I save anything. Each check is something that *should* be true if the data is healthy:

- **Date range** — does it actually span 2014 → 2026?
- **Nulls** — any missing values hiding anywhere?
- **Missing days** — is the daily timeline continuous, or are there gaps?
- **OHLC validity** — for every row, `high` must be the largest and `low` the smallest. If `high < low`, the row is corrupt.
- **Volume consistency** — after my fix, does `volume_btc × close ≈ volume_usd` across the whole dataset?

In [11]:
print("=== VALIDATION ===")
print("Range :", merged["date"].min().date(), "->", merged["date"].max().date())
print("Rows  :", len(merged))
print("Nulls :", int(merged.isnull().sum().sum()))

full = pd.date_range(merged["date"].min(), merged["date"].max(), freq="D")
missing = set(full) - set(merged["date"])
print(f"Missing days: {len(missing)} of {len(full)} calendar days")

bad = merged[(merged["high"] < merged["low"]) |
             (merged["high"] < merged[["open", "close"]].max(axis=1)) |
             (merged["low"] > merged[["open", "close"]].min(axis=1))]
print("OHLC-invalid rows:", len(bad))

vol_err = ((merged["volume_btc"] * merged["close"] - merged["volume_usd"]).abs()
           / merged["volume_usd"].replace(0, 1)).median()
print(f"Median volume inconsistency: {vol_err:.4f}  (closer to 0 is better)")

=== VALIDATION ===
Range : 2014-11-28 -> 2026-06-16
Rows  : 4219
Nulls : 0
Missing days: 0 of 4219 calendar days
OHLC-invalid rows: 0
Median volume inconsistency: 0.0065  (closer to 0 is better)


If all of that came back clean (0 nulls, 0 missing days, 0 invalid OHLC, tiny volume error), I'm confident the dataset is solid.

## 9. Round the numbers and save

Last cosmetic step: prices to 2 decimals (cents), volumes to a sensible precision. Then write it to `data/processed/`.

In [12]:
for c in ["open", "high", "low", "close"]:
    merged[c] = merged[c].round(2)
merged["volume_btc"] = merged["volume_btc"].round(4)
merged["volume_usd"] = merged["volume_usd"].round(2)

merged.to_csv(OUT, index=False)
print("Saved ->", OUT)
print("\nFirst rows:")
print(merged.head(3).to_string(index=False))
print("\nLast rows:")
print(merged.tail(3).to_string(index=False))

Saved -> D:\Bitcoin-Trend-Prediction\data\processed\btc_usd_daily_2014_2026.csv

First rows:
      date   open   high    low  close  volume_btc  volume_usd        source
2014-11-28 363.59 381.34 360.57 376.28     8617.15  3220878.18 early-history
2014-11-29 376.42 386.60 372.25 376.72     7245.19  2746157.05 early-history
2014-11-30 376.57 381.99 373.32 373.34     3046.33  1145566.61 early-history

Last rows:
      date     open     high      low    close  volume_btc   volume_usd  source
2026-06-14 64458.01 65800.00 63678.83 65746.45  13203.0338 8.536545e+08 binance
2026-06-15 65746.45 67292.15 65354.00 66328.74  18559.8010 1.230675e+09 binance
2026-06-16 66328.74 66463.26 65650.00 66241.66   4220.2761 2.788114e+08 binance


## 10. Recap

**What I started with:** two raw files, 9 and 12 columns, different names, overlapping years, and a swapped-volume bug in the old one.

**What I did:**
1. Inspected both files before touching them.
2. Decided which columns to keep — dropped redundant ones (`unix`, `symbol`, `Close time`, `Ignore`) and Binance-only ones (`Number of trades`, the taker volumes) that couldn't span the full history.
3. Fixed the swapped volume columns in the old file using the `USD ≈ BTC × price` rule.
4. Used Binance for 2018+ and the old file only for 2014–2017, so no overlap conflicts.
5. Merged, de-duplicated, sorted, and validated every assumption.

**What I ended up with:** one clean file, `data/processed/btc_usd_daily_2014_2026.csv`, with 8 columns — `date, open, high, low, close, volume_btc, volume_usd, source` — and no gaps across ~4,200 days.

**Why only 8 columns:** because a merged dataset can only keep what *both* sources provide. The extra Binance columns were sacrificed so the timeline stays continuous and gap-free.

**Next step:** write the API fetcher to pull the newest daily candles (from where this file ends) in this *exact* schema, so fresh data appends cleanly — then start feature engineering.